# Neural Prototyping

# Google Colab Mounting

In [1]:
!rm -rf /content/credit-risk-modeling
!git clone https://github.com/BillyBrothers/credit-risk-modeling.git
!pip install -r /content/credit-risk-modeling/requirements.txt

import sys 
sys.path.append('/content/credit-risk-modeling')

Cloning into 'credit-risk-modeling'...
remote: Enumerating objects: 1209, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 1209 (delta 59), reused 56 (delta 29), pack-reused 1121 (from 1)
Receiving objects: 100% (1209/1209), 52.28 MiB | 22.65 MiB/s, done.
Resolving deltas: 100% (821/821), done.
Updating files: 100% (68/68), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.0/230.0 kB 5.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.0 MB/s eta 0:00:00


In [2]:
!pip install scikeras

In [3]:
pip install -q -U keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 2.1 MB/s eta 0:00:00a 0:00:01


In [40]:
# data analysis
import pandas as pd
import numpy as np

# visualization
import matplotlib.pyplot as plt
import seaborn as sns
from pyampute.exploration.md_patterns import mdPatterns
from pyampute.exploration.mcar_statistical_tests import MCARTest
import missingno as msno

# preprocessing
import sklearn.utils.validation
import sys
from scipy import stats
from scipy.stats import shapiro, distributions, loguniform
from scipy.stats.mstats import winsorize
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import train_test_split, GridSearchCV, HalvingRandomSearchCV, HalvingGridSearchCV, TunedThresholdClassifierCV, FixedThresholdClassifier
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, PowerTransformer, QuantileTransformer, MinMaxScaler, KBinsDiscretizer, Binarizer, PolynomialFeatures, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from feature_engine.outliers import Winsorizer
from imblearn.over_sampling import SMOTE, SMOTENC
from sklearn.pipeline import Pipeline
from sklearn import set_config

# Feature Selection
from sklearn.feature_selection import SelectFromModel

# Modeling
from sklearn.linear_model import RidgeClassifier, LogisticRegression, RidgeClassifierCV, LogisticRegressionCV, SGDClassifier, Perceptron, PassiveAggressiveClassifier
from sklearn.svm import LinearSVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.dummy import DummyClassifier
import joblib

# Metrics
from sklearn.metrics import confusion_matrix, recall_score, precision_score, balanced_accuracy_score, ConfusionMatrixDisplay, classification_report, precision_recall_curve, PrecisionRecallDisplay, log_loss, brier_score_loss, roc_curve, roc_auc_score, RocCurveDisplay, det_curve, DetCurveDisplay, fbeta_score, average_precision_score, matthews_corrcoef

# Calibration
from sklearn.calibration import calibration_curve, CalibrationDisplay, CalibratedClassifierCV

# Inspection
from sklearn.inspection import PartialDependenceDisplay
from credit_risk_modeling import model_eval

import tensorflow as tf
from tensorflow import keras
from keras import layers
from scikeras.wrappers import KerasClassifier
import keras_tuner as kt

In [5]:
!ls

credit-risk-modeling  sample_data


In [6]:
!ls credit-risk-modeling

credit_risk_modeling  LICENSE	 pyproject.toml  requirements.txt
data		      Makefile	 README.md	 tests
docs		      models	 references
environment.yml       notebooks  reports


# Imports

In [7]:
X_train = pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/processed/X_train_neural.csv"
)

In [8]:
X_test = pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/processed/X_test_neural.csv"
)

In [9]:
X_val = pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/processed/X_val_neural.csv"
)

In [10]:
y_train= pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/interim/y_train.csv"
)
y_train = y_train.values.ravel()
neg, pos = np.bincount(y_train)
total = neg + pos
print(f"Examples:\n The training sets total amount of samples: {total}\n Positive: {pos} ({pos/total*100:.2f}% of total)")

Examples:
 The training sets total amount of samples: 22686
 Positive: 4962 (21.87% of total)


In [11]:
y_test = pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/interim/y_test.csv"
)
y_test = y_test.values.ravel()
neg, pos = np.bincount(y_test)
total = neg + pos
print(f"Examples:\n The testing set total amount of samples: {total}\n Positive: {pos} ({pos/total*100:.2f}% of total)")

Examples:
 The testing set total amount of samples: 2917
 Positive: 638 (21.87% of total)


In [12]:
y_val = pd.read_csv(
    filepath_or_buffer= "/content/credit-risk-modeling/data/interim/y_val.csv"
)
y_val = y_val.values.ravel()
neg, pos = np.bincount(y_val)
total = neg + pos
print(f"Examples:\n The validation set total amount of samples: {total}\n Positive: {pos} ({pos/total*100:.2f}% of total)")

Examples:
 The validation set total amount of samples: 6806
 Positive: 1488 (21.86% of total)


# Build Sequential Models

In [ ]:
# # Architecture 1: Single hidden dense layer
def mlp1():
    model = keras.Sequential(name='MLP-1')
    model.add(keras.Input(shape=(X_train.shape[1], ))),
    model.add(layers.Dense(units=64, activation='relu')),
    model.add(layers.Dropout(rate= 0.20)),
    model.add(layers.Dense(units=1,activation='sigmoid')),
    model.compile(
        optimizer= keras.optimizers.Adam(),
        loss= keras.losses.BinaryCrossentropy(),
        metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
)
    return model

In [15]:
# def mlp1(hp):
#     model1 = keras.Sequential(name='MLP-1')
#     model1.add(keras.Input(shape=(X_train.shape[1], )))

#     hp_units= hp.Int('units', min_value=64, max_value=512, step=32)
#     hp_lr = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')

#     model1.add(layers.Dense(units=hp_units, activation='relu'))
#     model1.add(layers.Dropout(rate= 0.20))
#     model1.add(layers.Dense(units=1,activation='sigmoid'))
    
#     model1.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_lr),
#                 loss=keras.losses.BinaryCrossentropy(),
#                 metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
#     ) 
#     return model1

In [97]:
# Architecture 2: Two hidden dense layers
def mlp2():
    model = keras.Sequential(name='MLP-2')
    model.add(keras.Input(shape=(X_train.shape[1], ))),
    model.add(layers.Dense(units=64, activation='relu')),
    model.add(layers.Dropout(rate= 0.20)),
    model.add(layers.Dense(units=128, activation='relu')),
    model.add(layers.Dropout(rate=0.20)),
    model.add(layers.Dense(units=1,activation='sigmoid')),
    
    model.compile(optimizer=keras.optimizers.Adam(),
                loss=keras.losses.BinaryCrossentropy(),
                metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
    )
    return model

In [17]:
# # Architecture 2: Two hidden dense layers
# def mlp2(hp):
#     model2 = keras.Sequential(name='MLP-2')
#     model2.add(keras.Input(shape=(X_train.shape[1], ))),
#     hp_units= hp.Int('units', min_value=64, max_value=512, step=32)
#     model2.add(layers.Dense(units=hp_units, activation='relu')),
#     model2.add(layers.Dropout(rate= 0.20)),
#     hp_units= hp.Int('units', min_value=64, max_value=512, step=32)
#     model2.add(layers.Dense(units=hp_units, activation='relu')),
#     model2.add(layers.Dropout(rate=0.20)),
#     model2.add(layers.Dense(units=1,activation='sigmoid')),
#     model2.compile(optimizer=keras.optimizers.Adam(),
#                 loss=keras.losses.BinaryCrossentropy(),
#                 metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
#     )
#     return model2

In [18]:
# Architecture 3: Three hidden dense layers
def mlp3():
    model = keras.Sequential(name='MLP-3')
    model.add(keras.Input(shape=(X_train.shape[1], ))),
    model.add(layers.Dense(units=64, activation='relu')),
    model.add(layers.Dropout(rate= 0.20)),
    model.add(layers.Dense(units=128, activation='relu')),
    model.add(layers.Dropout(rate=0.20)),
    model.add(layers.Dense(units=256, activation='relu')),
    model.add(layers.Dropout(rate=0.2)),
    model.add(layers.Dense(units=1,activation='sigmoid')),

    model.compile(
        optimizer= keras.optimizers.Adam(),
        loss= keras.losses.BinaryCrossentropy(),
        metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
)
    return model

In [ ]:
# def mlp3(hp):
#     model3 = keras.Sequential(name='MLP-3')
#     model3.add(keras.Input(shape=(X_train.shape[1], ))),
#     hp_units= hp.Int('units', min_value=64, max_value=512, step=32)
#     model3.add(layers.Dense(units=hp_units, activation='relu')),
#     model3.add(layers.Dropout(rate= 0.20)),
#     hp_units= hp.Int('units', min_value=64, max_value=512, step=32)
#     model3.add(layers.Dense(units=hp_units, activation='relu')),
#     model3.add(layers.Dropout(rate=0.20)),
#     hp_units= hp.Int('units', min_value=64, max_value=512, step=32)
#     model3.add(layers.Dense(units=hp_units, activation='relu')),
#     model3.add(layers.Dropout(rate=0.2)),
#     model3.add(layers.Dense(units=1,activation='sigmoid')),
#     model3.compile(optimizer=keras.optimizers.Adam(),
#                 loss=keras.losses.BinaryCrossentropy(),
#                 metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
#     )

#     return model3

In [98]:
mlp_models = [
    ("MLP-1", mlp1()),
    ("MLP-2", mlp2()),
    ("MLP-3", mlp3())
]

### Callbacks

In [ ]:
reduce_lr_plateau = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    verbose=1,
    min_lr=0.001
)

In [22]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    min_delta=1e-4,
    patience=7,
    verbose=1,
    restore_best_weights=True
)

In [23]:
log_dir = "logs/fit/"
tensorboard = keras.callbacks.TensorBoard(
    log_dir= log_dir
)

### Fit

In [24]:
class_weight = {
    0: 1.0,
    1: 2.0
    }

In [25]:
histories = {}

In [26]:
for model_name, model in mlp_models:
    print(f"Currently fitting model {model_name}.")
    history = model.fit(
        x= X_train,
        y= y_train,
        batch_size=32,
        epochs= 100,
        verbose=2,
        callbacks= [early_stopping, reduce_lr_plateau, tensorboard],
        validation_data= (X_val, y_val),
        class_weight= class_weight
    )
    histories[model_name] = history.history

Currently fitting model MLP-1.
Epoch 1/100
709/709 - 3s - 4ms/step - auc: 0.8547 - loss: 0.5480 - val_auc: 0.8928 - val_loss: 0.3365 - learning_rate: 1.0000e-03
Epoch 2/100
709/709 - 1s - 2ms/step - auc: 0.8888 - loss: 0.4807 - val_auc: 0.9029 - val_loss: 0.3268 - learning_rate: 1.0000e-03
Epoch 3/100
709/709 - 1s - 2ms/step - auc: 0.8950 - loss: 0.4639 - val_auc: 0.9065 - val_loss: 0.3190 - learning_rate: 1.0000e-03
Epoch 4/100
709/709 - 1s - 2ms/step - auc: 0.8992 - loss: 0.4527 - val_auc: 0.9079 - val_loss: 0.3032 - learning_rate: 1.0000e-03
Epoch 5/100
709/709 - 2s - 3ms/step - auc: 0.9030 - loss: 0.4425 - val_auc: 0.9101 - val_loss: 0.2994 - learning_rate: 1.0000e-03
Epoch 6/100
709/709 - 1s - 2ms/step - auc: 0.9032 - loss: 0.4400 - val_auc: 0.9100 - val_loss: 0.3029 - learning_rate: 1.0000e-03
Epoch 7/100
709/709 - 1s - 2ms/step - auc: 0.9054 - loss: 0.4334 - val_auc: 0.9124 - val_loss: 0.3069 - learning_rate: 1.0000e-03
Epoch 8/100
709/709 - 1s - 2ms/step - auc: 0.9065 - loss: 0

In [27]:
chosen_metric = 'val_auc'
max_auc_per_model = {}
best_auc = None
best_model = None
best_model_name = None

for model_name, model in mlp_models:
    max_auc_per_model[model_name] = max(histories[model_name][chosen_metric])
    max_auc_dict = dict([sorted(max_auc_per_model.items(), key= lambda item: item[1])[-1]])
    max_auc_model_name = list(max_auc_dict.keys())[0]
    max_aux_score = list(max_auc_dict.values())[0]
    if max_auc_model_name == model_name:
        best_auc = max_aux_score
        best_model_name = max_auc_model_name
        best_model = model
    else:
        continue

print(f"Results: \nBest model: {best_model_name}\nAUC score: {best_auc:.4f}")

Results: 
Best model: MLP-2
AUC score: 0.9260


In [28]:
best_model.summary()

Model: "MLP-2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 64)             │         1,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,189 (114.02 KB)

 Trainable params: 9,729 (38.00 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 19,460 (76.02 KB)

### Hyperparameter Tuning

In [29]:
# for tid, t in tuner.oracle.trials.items():
#     print(tid, t.status, t.score)

In [30]:
# def mlp2_tuned(hp):
#     model = keras.Sequential(name='MLP-2-tuned')

#     units = hp.Int("units", 64, 256, step=32)
#     lr = hp.Float("learning_rate", 1e-4, 1e-2, sampling="log")

#     model.add(keras.Input(shape=(X_train.shape[1], ))),
#     model.add(layers.Dense(units=units, activation='relu')),
#     model.add(layers.Dropout(rate= 0.20)),
#     model.add(layers.Dense(units=units, activation='relu')),
#     model.add(layers.Dropout(rate=0.20)),
#     model.add(layers.Dense(units=1,activation='sigmoid')),
    
#     model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
#                 loss=keras.losses.BinaryCrossentropy(),
#                 metrics= [keras.metrics.AUC(curve="ROC", name="auc")]
#     )
#     return model

In [31]:
# import shutil
# shutil.rmtree("untitled_project", ignore_errors=True)

In [32]:
# tuner = kt.Hyperband(
#     hypermodel= mlp2_tuned,
#     objective="val_auc",
#     max_epochs= 50,
#     factor=3,
# )

In [33]:
# tuner.search_space_summary()

In [34]:
# tuner.search(
#     X_train,
#     y_train,
#     validation_data= (X_val, y_val),
#     callbacks= [early_stopping]
# )

In [35]:
# tuner.results_summary()

### Calibration

Scikeras cannot accept an instance of a model only a user building function (factory function). So, my model will have NO weights on it.

In [140]:
models_only = {}
mlp_functions = []

In [141]:
for model_name, model in mlp_models:
    models_only[model_name] = KerasClassifier(
    model= model,
    optimizer= keras.optimizers.Adam(),
    loss= keras.losses.BinaryCrossentropy(),
    random_state=42,
    class_weight= class_weight,
    metrics= ['val_auc'],
    callbacks= [early_stopping, reduce_lr_plateau],
    validation_split= 0.20,
    epochs=100
)

In [142]:
len(models_only)

3

In [150]:
for model_name, model in list(models_only.items()):
    mlp_functions.append(model)

In [157]:
type(mlp_functions[0])

scikeras.wrappers.KerasClassifier

In [152]:
model_eval.comparing_models(
    mlp_functions,
    X_train,
    y_train,
    X_test,
    y_test
)

Epoch 1/100
568/568 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - auc: 0.7903 - loss: 0.6212 - val_auc: 0.8900 - val_loss: 0.4938 - learning_rate: 0.0010
Epoch 2/100
568/568 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - auc: 0.8804 - loss: 0.4875 - val_auc: 0.8998 - val_loss: 0.4684 - learning_rate: 0.0010
Epoch 3/100
568/568 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - auc: 0.8904 - loss: 0.4654 - val_auc: 0.9044 - val_loss: 0.4555 - learning_rate: 0.0010
Epoch 4/100
568/568 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - auc: 0.8916 - loss: 0.4605 - val_auc: 0.9073 - val_loss: 0.4477 - learning_rate: 0.0010
Epoch 5/100
568/568 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - auc: 0.8964 - loss: 0.4486 - val_auc: 0.9088 - val_loss: 0.4425 - learning_rate: 0.0010
Epoch 6/100
568/568 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - auc: 0.8977 - loss: 0.4446 - val_auc: 0.9098 - val_loss: 0.4387 - learning_rate: 0.0010
Epoch 7/100
568/568 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - auc: 0.8996 - loss: 0.4391 - val_auc: 0.9109 - val_loss: 0.4342 - learning_rate: 0.0010

(                                             model   roc_auc    pr_auc  \
 1  KerasClassifier (class_weight={0: 1.0, 1: 2.0})  0.920762  0.859077   
 2  KerasClassifier (class_weight={0: 1.0, 1: 2.0})  0.917324  0.853402   
 0  KerasClassifier (class_weight={0: 1.0, 1: 2.0})  0.915895  0.849064   
 
    log_loss  brier_score  matthews_corrcoef  
 1  0.252541     0.073231           0.735568  
 2  0.256958     0.074694           0.730853  
 0  0.265793     0.077368           0.717380  ,
 {'KerasClassifier (class_weight={0: 1.0, 1: 2.0})': KerasClassifier(
  	model=<Sequential name=MLP-3, built=True>
  	build_fn=None
  	warm_start=False
  	random_state=42
  	optimizer=<keras.src.optimizers.adam.Adam object at 0x7b9eba4a9310>
  	loss=<LossFunctionWrapper(<function binary_crossentropy at 0x7b9edf75de40>, kwargs={'from_logits': False, 'label_smoothing': 0.0, 'axis': -1})>
  	metrics=['val_auc']
  	batch_size=None
  	validation_batch_size=None
  	verbose=1
  	callbacks=[<keras.src.callbacks.

In [ ]:
fitted_models